In [1]:
#| label: setup
#| include: false

from pathlib import Path
import sys
import json
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "analyst").exists() and (candidate / "config").exists():
            return candidate
    raise RuntimeError("Repo root not found from notebook working directory.")

_root = find_repo_root(Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from analyst.table_formatting import display_analysis_table

SESSION_ID = "strat_solusdt_fw60_combo_2101_2605"
artifact_path = _root / "artifacts" / SESSION_ID / "strategy_artifact.json"
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
trades_df = pd.read_csv(_root / "artifacts" / SESSION_ID / artifact["trade_csv_path"], parse_dates=["entry_time", "exit_time"])
summary_df = pd.read_csv(_root / "artifacts" / SESSION_ID / artifact["summary_csv_path"])

trade_report_df = trades_df.loc[:, [
    "entry_time",
    "exit_time",
    "entry_price",
    "exit_price",
    "profit_pct",
    "expected_log_return",
    "fact_log_return",
    "fact_1h_max_range_log_return",
    "exit_reason",
]].copy()


## Logika

Ez a riport a jelenlegi baseline szabályt mutatja:
csak `long`, csak `D10`, belépés `score_pct_long >= 0.90`,
take profit a D10 medián 1 órás target log returnjén,
stop loss ennek szimmetrikus negatívja,
különben zárás 60 perc után az aktuális close áron.
TP vagy SL utáni korai zárás után a következő bartól újra beléphet.


In [2]:
#| label: tbl-trade-ledger
#| tbl-cap: "Trade ledger: entry, exit, várakozás és tény"

display_analysis_table(trade_report_df)


In [3]:
#| label: tbl-summary
#| tbl-cap: "Összefoglaló exit reason szerint"

display_analysis_table(summary_df)


exit_reason,n_trades,avg_entry_price,avg_exit_price,avg_profit_pct,avg_expected_log_return,avg_fact_log_return,avg_fact_1h_max_range_log_return,total_fact_log_return,compounded_return_pct,realized_directional_win_rate,avg_hold_minutes,tp_target_log_return
stop_loss,1289,99.170,98.567,-0.61%,0.006,-0.006,0.005,-7.868,-1.00%,0.00%,14.712,0.006
take_profit,1263,98.538,99.141,0.61%,0.006,0.006,0.014,7.709,2227.07%,100.00%,13.985,0.006
timeout_1h,204,100.538,100.566,0.03%,0.006,0.000,0.003,0.066,0.07%,53.92%,60.000,0.006
ALL,2756,98.982,98.978,-0.00%,0.006,-0.000,0.009,-0.093,-0.09%,49.82%,17.731,0.006
